## Apartado 2. Clasificador de comentarios en subreddits

En esta sección vamos a desarrollar y analizar distintos modelos de procesamiento del lenguaje para clasificación. Nuestro clasificador será capaz de recibir un comentario y predecir a que subreddit de los 6 seleccionados (books, jobs, LeagueOfLegends, RandomThoughts, travel, unpopopularopinion) pertenece el mensaje, estimando las probabilidades de pertenecer a cada uno de ellos.

Con el fin de obtener un modelo robusto e insesgado, vamos a realizar en primer lugar la división en conjunto de entrenamiento y validación.

Para ello utilizaremos los ficheros JSON limpios generados en el apartado anterior. Para la división vamos a utilizar una estrategia basada en hilos y no en comentarios individuales.

El ratio de división entre entrenamiento y validación, va a ser 70% y 30%, respectivamente. 

Como ya hemos mencionado, con el fin de que nuestro modelo generalice correctamente y no memorice patrones concretos de un determinado hilo, todos los comentarios de un mismo hilo se asignará únicamente a uno de los dos conjuntos, o bien el de entrenamiento, o bien al de validación, pero nunca a ambos. En el apartado anterior hemos obtenido 59 hilos por cada subreddit, por lo que utilizaremos aproximadamente 41 hilos para entrenar y 18 para validar. Teniendo 20 comentarios por hilo, contaremos con un total de:
- 6 (subreddits) * 41 (hilos) * 20 (comentarios) = 4920 comentarios en el conjunto de entrenamiento
- 6 (subreddits) * 18 (hilos) * 20 (comentarios) = 2160 comentarios en el conjunto de validación

In [1]:
import json
import regex as re
import random

subreddits = ["limpio_books.json", "limpio_jobs.json", "limpio_LeagueOfLegends.json", "limpio_RandomThoughts.json",
			  "limpio_travel.json", "limpio_unpopularopinion.json"]

X_train, X_test = [], []
y_train, y_test = [], []

proporcion = 0.7

for subreddit in subreddits:
	with open(subreddit, "r", encoding="utf-8") as file:
		datos = json.load(file)

	# Obtenemos el nombre del subreddit, ya que será nuestra salida esperada "y"
	nombre = datos["subreddit"]

	# Extraemos todos los hilos pertenecientes al subreddit
	hilos = datos["submissions"]

	# Mezclamos los hilos para que la división sea aleatoria
	random.shuffle(hilos)

	# Como hemos calculado en la explicación, en base a los hilos que tengamos en cada subreddit, utilizaremos unos
	# para entrenamiento y otros para validación. Multiplicando por la proporcion obtendremos valores con decimales
	# por lo que lo pasamos a entero, ya que los índices son enteros.
	indice = int(len(hilos) * proporcion)

	# Extraemos los hilos que pertenecerán al conjunto de entrenamiento y los que pertenecerán a la validación
	hilos_train = hilos[:indice]
	hilos_test = hilos[indice:]	

	# Guardamos el texto de los comentarios en el conjunto train, aplicándole un preprocesamiento, en este caso pasar el texto a minúsculas, 
	# y el subreddit al que pertenece.
	for hilo in hilos_train:
		for comentario in hilo["comments"]:
			X_train.append(comentario["body"].lower())
			y_train.append(nombre)

	# Guardamos el texto de los comentarios en el conjunto test, aplicándole un preprocesamiento, en este caso pasar el texto a minúsculas, 
	# y el subreddit al que pertenece.
	for hilo in hilos_test:
		for comentario in hilo["comments"]:
			X_test.append(comentario["body"].lower())
			y_test.append(nombre)

In [2]:
# Comprobamos que el resultado obtenido es el que habíamos calculado previamente
print(f"Conjunto de entrenamiento: Hemos obtenido {len(X_train)} comentarios")
print(f"Conjunto de validación: Hemos obtenido {len(X_test)} comentarios")

Conjunto de entrenamiento: Hemos obtenido 4920 comentarios
Conjunto de validación: Hemos obtenido 2160 comentarios


Una vez hemos obtenido los conjuntos necesarios para continuar con el siguiente paso donde usaremos distintos tipos de representación de textos para hacer esta clasificación.
# ... escribir explicación inicial

In [3]:
import nltk
# Descargamos las stopwords de NLTK
# Si no tenemos instalado NLTK lo instalamos
# !pip3 install -U nltk
nltk.download('stopwords')

stopwords = nltk.corpus.stopwords.words('english')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/daniela/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report,  accuracy_score

# Se pueden crear modelos mediante Pipeline
# Creamos el pipeline de TF con LinearSVC
clf_tf = Pipeline([
    ('vect', CountVectorizer(stop_words=stopwords)),
    ('tf', TfidfTransformer(use_idf=False)),
    ('clf', RandomForestClassifier(random_state=23)),])

clf_tf.fit(X_train, y_train)

y_pred = clf_tf.predict(X_test)


report = classification_report(y_test, y_pred)
print(report)

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

                  precision    recall  f1-score   support

 LeagueOfLegends       0.83      0.57      0.68       360
  RandomThoughts       0.31      0.62      0.41       360
           books       0.73      0.57      0.64       360
            jobs       0.61      0.53      0.56       360
          travel       0.73      0.61      0.66       360
unpopularopinion       0.35      0.29      0.32       360

        accuracy                           0.53      2160
       macro avg       0.59      0.53      0.54      2160
    weighted avg       0.59      0.53      0.54      2160

0.5296296296296297


# pequeña reflexion de resultados

In [8]:
#Import all the dependencies
from gensim.models.doc2vec import TaggedDocument

#Necestiamos crear un TaggedDocument para cada uno de los textos indicando un índice de cada texto
# Es necesario el indice, creamos uno para cada texto de entrenamiento
tagged_data = [TaggedDocument(words=datos, tags=[str(i)])
               for i, datos in enumerate(X_train)]

In [10]:
from gensim.models import Word2Vec

model = Word2Vec(X_train, vector_size=100, window=10, min_count=1, workers=10)

In [11]:
from gensim.models import Doc2Vec

# Definimos los parámetros de entrenamiento y entrenamos
max_epochs = 5
vec_size = 100
alpha = 0.025

doc2vec_model = Doc2Vec(vector_size=vec_size,
                alpha=alpha,
                min_alpha=0.00025,
                min_count=1,
                dm = 1,
                epochs = max_epochs)

doc2vec_model.build_vocab(tagged_data)

for epoch in range(max_epochs):
    doc2vec_model.train(tagged_data,
                total_examples=doc2vec_model.corpus_count,
                epochs=doc2vec_model.epochs)
    # decrease the learning rate
    doc2vec_model.alpha -= 0.0002
    # fix the learning rate, no decay
    doc2vec_model.min_alpha = model.alpha

doc2vec_model.save("d2v.model")
print("Model Saved")

Model Saved


In [15]:
import nltk

nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/daniela/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [17]:
from nltk.tokenize import word_tokenize

def vectorizar_comentario(texto, modelo):
	tokens = word_tokenize(texto)
	return modelo.infer_vector(tokens)

X_train_d2v = [vectorizar_comentario(comentario, doc2vec_model) for comentario in X_train]
X_test_d2v = [vectorizar_comentario(comentario, doc2vec_model) for comentario in X_test]

In [18]:
from sklearn.svm import LinearSVC
from sklearn import metrics

clf_d2v = LinearSVC(random_state=23, tol=1e-15)
clf_d2v.fit(X_train_d2v, y_train)

y_pred_d2v = clf_d2v.predict(X_test_d2v)

accuracy = metrics.accuracy_score(y_test, y_pred_d2v)
print(accuracy)

report = classification_report(y_test, y_pred)
print(report)

/Users/daniela/anaconda3/lib/python3.11/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


0.25462962962962965
                  precision    recall  f1-score   support

 LeagueOfLegends       0.71      0.73      0.72       360
  RandomThoughts       0.38      0.41      0.39       360
           books       0.64      0.64      0.64       360
            jobs       0.59      0.58      0.59       360
          travel       0.68      0.70      0.69       360
unpopularopinion       0.37      0.31      0.34       360

        accuracy                           0.56      2160
       macro avg       0.56      0.56      0.56      2160
    weighted avg       0.56      0.56      0.56      2160



/Users/daniela/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:1242: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


# poner mas dimensiones en vec_size y hacer comparacion

In [19]:
# pip install transformers datasets evaluate accelerate
from transformers import AutoTokenizer

modelo = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(modelo)

/Users/daniela/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
import numpy as np
import evaluate

# 1. Elección del modelo de Hugging Face
# Usaremos 'distilbert-base-uncased' porque es rápido y eficiente
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# 2. Preparar los datos en el formato que requiere Hugging Face
# Convertimos nuestras listas X e y a un DataFrame y luego a un Dataset
df_train = pd.DataFrame({"text": X_train, "label": y_train})
df_test = pd.DataFrame({"text": X_test, "label": y_test})

# Es necesario mapear los nombres de los subreddits a números (0, 1, 2...)
label_map = {label: i for i, label in enumerate(subreddits)}
df_train["label"] = df_train["label"].map(label_map)
df_test["label"] = df_test["label"].map(label_map)

train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

# 3. Tokenización del corpus
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

# 4. Configuración del modelo para clasificación
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=len(subreddits)
)

# 5. Entrenamiento (Fine-tuning)
training_args = TrainingArguments(
    output_dir="resultados_bert",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3, # 3 épocas suelen ser suficientes para fine-tuning[cite: 8]
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
)

print("⏳ Iniciando el Fine-tuning del Transformer...")
trainer.train()

/Users/daniela/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/daniela/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/daniela/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

⏳ Iniciando el Fine-tuning del Transformer...


/Users/daniela/anaconda3/lib/python3.11/site-packages/transformers/optimization.py:407: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/924 [00:00<?, ?it/s]

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`label` in this case) have excessive nesting (inputs type `list` where type `int` is expected).